# ☸️ Lab 2: Kubernetes Fundamentals with Pods and ReplicaSets – Deploy a Flask App

🕒 **Estimated Time:** 25–30 Minutes  
🎯 **Focus Areas:** Kubernetes Architecture, Pod Lifecycle, ReplicaSets, Self-Healing, Declarative YAML, Scaling

---

## 🧾 Lab Overview
In this lab, you will deploy the containerized **Python-Flask application** on a Kubernetes cluster. This lab introduces the foundational building blocks of Kubernetes:
- **Pod:** The smallest deployable computing unit in Kubernetes, wrapping one or more containers.
- **ReplicaSet:** A controller ensuring a specified number of identical Pod replicas are running at all times.

```
       +-------------------------------------------------------+
       |                  ReplicaSet Controller                |
       |                   (replicas: 3)                       |
       +-------------------------------------------------------+
                |                    |                    |
                v                    v                    v
         +-------------+      +-------------+      +-------------+
         |     Pod     |      |     Pod     |      |     Pod     |
         | [flask-app] |      | [flask-app] |      | [flask-app] |
         +-------------+      +-------------+      +-------------+
```

---

## 🎯 Objectives
- 🚀 Deploy a Python-Flask app as a standalone Pod
- 🔍 Inspect Pod status, describe events, and view logs
- 🐚 Connect to a running Pod container using `kubectl exec`
- 🔄 Deploy a ReplicaSet to ensure high availability
- 🛡️ Demonstrate Kubernetes self-healing by deleting a managed Pod
- 📈 Scale workloads up and down dynamically
- 🧹 Clean up resources safely


---
## ☘️ Step 1: Verify Kubernetes Cluster Status
Ensure your Minikube or Kubernetes cluster is running and `kubectl` can communicate with the control plane.


In [ ]:
!kubectl cluster-info


In [ ]:
!kubectl get nodes -o wide


> **Tip:** If nodes are not ready or Minikube is stopped, start it in terminal using `minikube start`.


---
## 🧩 PART 1: Deploy Flask App as a Pod

### ☘️ Step 1.1: Explore Pod YAML Manifest
Open and review `Lab2/flask_pod.yaml`. Let's print its content:


In [ ]:
with open('flask_pod.yaml', 'r') as f:
    print(f.read())


### 🔍 Key Manifest Fields:
- `apiVersion: v1`: Core Kubernetes API group.
- `kind: Pod`: Declares this object as a Pod.
- `metadata.name: flask-app`: Unique identifier within the namespace.
- `metadata.labels.app: flask`: Key-value pair used by selectors and queries.
- `spec.containers`: Array specifying container image (`technoavengers/flask-app:v1`) and target port `5000`.


---
### ☘️ Step 1.2: Deploy the Pod
Apply the manifest to your cluster using `kubectl apply`.


In [ ]:
!kubectl apply -f flask_pod.yaml


---
### ☘️ Step 1.3: Verify Pod Status
Check the status of the newly created Pod. Look for the `STATUS` column (`Pending` -> `ContainerCreating` -> `Running`).


In [ ]:
!kubectl get pods -o wide


---
### ☘️ Step 1.4: Deep Dive Inspection with `kubectl describe`
Inspect detailed pod metadata, node placement, IP assignment, and lifecycle events.


In [ ]:
!kubectl describe pod flask-app


> 💡 **What to observe:** Look at the bottom **Events** section. You will see scheduler assignments, container image pulling, and container start events.


---
### ☘️ Step 1.5: View Pod Logs
Fetch stdout logs from the Flask application container.


In [ ]:
!kubectl logs flask-app


---
### ☘️ Step 1.6: Execute Commands Inside the Pod Shell
Use `kubectl exec` to run commands inside the container environment.


In [ ]:
# Inspect processes inside the container
!kubectl exec flask-app -- ps aux


In [ ]:
# List files in application directory
!kubectl exec flask-app -- ls -la /python-flask


> 💡 **Interactive Shell:** In a terminal, you can enter an interactive shell with:
> ```bash
> kubectl exec -it flask-app -- /bin/bash
> ```


---
### ☘️ Step 1.7: Delete the Standalone Pod
Delete the standalone Pod to observe that standalone pods are not automatically recreated when terminated.


In [ ]:
!kubectl delete pod flask-app


In [ ]:
# Verify pod termination
!kubectl get pods


---
## 🧩 PART 2: Deploy Flask App with ReplicaSet

### ☘️ Step 2.1: Explore ReplicaSet YAML Manifest
A ReplicaSet maintains a stable set of replica Pods running at any given time.
Let's inspect `Lab2/flask_replicaset.yaml`:


In [ ]:
with open('flask_replicaset.yaml', 'r') as f:
    print(f.read())


### 🔍 Key ReplicaSet Fields:
- `apiVersion: apps/v1`: Apps API group for workload controllers.
- `kind: ReplicaSet`: Controller managing pod replicas.
- `spec.replicas: 3`: Desired number of pod instances.
- `spec.selector.matchLabels`: Label selector identifying pods belonging to this ReplicaSet.
- `spec.template`: Pod template used to spawn new pods when current count < desired count.


---
### ☘️ Step 2.2: Deploy the ReplicaSet
Apply the ReplicaSet manifest:


In [ ]:
!kubectl apply -f flask_replicaset.yaml


---
### ☘️ Step 2.3: Verify ReplicaSet and Managed Pods
Check the ReplicaSet controller status and observe the 3 pods created automatically.


In [ ]:
!kubectl get replicaset flask-replicaset


In [ ]:
!kubectl get pods -l app=flask -o wide


---
### ☘️ Step 2.4: Test Self-Healing (Resilience)
Let's simulate a pod crash or node failure by deleting one of the pods managed by the ReplicaSet.
Observe how the ReplicaSet controller detects the discrepancy (`current < desired`) and immediately schedules a replacement!


In [ ]:
import subprocess

# Get the name of one running pod
output = subprocess.check_output(["kubectl", "get", "pods", "-l", "app=flask", "-o", "jsonpath={.items[0].metadata.name}"]).decode().strip()
print(f"Target Pod to delete: {output}")

# Delete the pod
!kubectl delete pod {output}


In [ ]:
# Verify that a brand new pod was automatically created
!kubectl get pods -l app=flask


---
### ☘️ Step 2.5: Scale Up the ReplicaSet
Scale the desired replica count from `3` to `5`.


In [ ]:
!kubectl scale replicaset flask-replicaset --replicas=5


In [ ]:
!kubectl get replicaset flask-replicaset
!kubectl get pods -l app=flask


---
### ☘️ Step 2.6: Scale Down the ReplicaSet
Scale down the workload from `5` to `2` replicas.


In [ ]:
!kubectl scale replicaset flask-replicaset --replicas=2


In [ ]:
!kubectl get replicaset flask-replicaset
!kubectl get pods -l app=flask


---
### ☘️ Step 2.7: Clean Up ReplicaSet
Delete the ReplicaSet. This will also cascade-delete all pods managed by it.


In [ ]:
!kubectl delete replicaset flask-replicaset


In [ ]:
# Verify all pods are terminating or removed
!kubectl get pods -l app=flask


---
## 📊 Summary & Comparison

| Feature | Standalone Pod | ReplicaSet | Deployment (Future Lab) |
| :--- | :--- | :--- | :--- |
| **Self-Healing** | ❌ No | ✅ Yes | ✅ Yes |
| **Multi-Replica Scaling** | ❌ No | ✅ Yes | ✅ Yes |
| **Declarative Updates** | ❌ Manual | ⚠️ Limited | ✅ Rolling Updates & Rollbacks |
| **Production Use** | Debugging only | Managed by Deployments | ✅ Recommended |

---

## ✅ Conclusion
Congratulations! You have completed **Lab 2**:
- Deployed standalone Pods and inspected the Pod lifecycle
- Understood and configured ReplicaSets with label selectors
- Observed automated self-healing and dynamic scaling
- Learned best practices for workload lifecycle management

👉 **Next Step:** Head to **Lab 3** to expose your Flask application with Kubernetes Services (`NodePort`)!
